# Notebook 05: Flan-T5 Fine-Tuning for Answer Evaluation

This notebook builds a lightweight training dataset from the NCERT answer bank and fine-tunes Flan-T5-small with LoRA for marks and feedback generation.

In [ ]:
from pathlib import Path
import json
import sys
import random

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = "google/flan-t5-small"
OUTPUT_DIR = REPO_ROOT / "models" / "flan_t5_small"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

answer_bank_path = REPO_ROOT / "data" / "answer_bank" / "ncert_answers.json"
if not answer_bank_path.exists():
    raise FileNotFoundError(f"Answer bank not found: {answer_bank_path}")

answer_bank = json.loads(answer_bank_path.read_text(encoding="utf-8"))
print(f"Loaded answer bank from {answer_bank_path}")

In [ ]:
def flatten_answer_bank(answer_bank):
    rows = []
    for subject, chapters in answer_bank.items():
        for chapter_name, questions in chapters.items():
            for question_key, payload in questions.items():
                rows.append({
                    "subject": subject,
                    "chapter": chapter_name,
                    "question_key": question_key,
                    "question": payload.get("question", ""),
                    "answer": payload.get("answer", ""),
                    "marks": payload.get("marks", 5),
                })
    return rows

rows = flatten_answer_bank(answer_bank)
print(f"Flattened {len(rows)} question-answer entries")

# Build a compact but meaningful instruction-tuning dataset.
# Each sample teaches the model to output a score and feedback for a student answer.
examples = []
for row in rows[:120]:
    subject = row["subject"]
    question = row["question"]
    expected = row["answer"]
    marks = row["marks"]

    variants = [
        (expected, marks, "Excellent answer. You explained the main idea clearly."),
        (expected.split()[0] if expected.split() else expected, max(1, marks - 2), "You gave a partial answer. Add a few more details to improve your score."),
        ("I do not know", 0, "This answer needs more effort and a clearer understanding of the topic."),
    ]

    for student_answer, sample_marks, feedback in variants:
        prompt = (
            f"Subject: {subject}\n"
            f"Question: {question}\n"
            f"Expected Answer: {expected}\n"
            f"Student Answer: {student_answer}\n"
            "Evaluate the student's answer. Return only: marks out of 5, feedback"
        )
        response = f"{sample_marks}/5 | {feedback}"
        examples.append((prompt, response))

random.seed(42)
random.shuffle(examples)

print(f"Prepared {len(examples)} training examples")
examples[:3]

In [ ]:
max_examples = min(len(examples), 240)
train_examples = examples[:max_examples]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def build_dataset(samples):
    prompts = [prompt for prompt, _ in samples]
    responses = [response for _, response in samples]
    tokenized_inputs = tokenizer(prompts, truncation=True, padding=True, max_length=384)
    tokenized_outputs = tokenizer(responses, truncation=True, padding=True, max_length=128)
    return Dataset.from_dict({
        "input_ids": tokenized_inputs["input_ids"],
        "attention_mask": tokenized_inputs["attention_mask"],
        "labels": tokenized_outputs["input_ids"],
    })

train_dataset = build_dataset(train_examples)
print(train_dataset)

In [ ]:
# Use a conservative LoRA setup that is stable on Colab GPUs.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

trainer.train()


# Save the base model + LoRA adapter so the app can load them later without retraining.
print(f"Saved fine-tuned model to {OUTPUT_DIR}")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
sample_prompt = train_examples[0][0]
encoded = tokenizer(sample_prompt, return_tensors="pt", truncation=True, max_length=384)
with torch.no_grad():
    output_ids = model.generate(**encoded, max_new_tokens=80)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

## Notes

For a full academic project run, use the full 600-sample dataset and a GPU-enabled Colab runtime to obtain stronger grading quality.